In [0]:
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    TimestampType
)

# ============================================================
# 1. DONNÉES CARRIERS
# ============================================================

carriers = [
    (
        "C001",
        "Atlas Express",
        "Express",
        "Morocco",
        "Premium",
        True,
        datetime(2026, 1, 1, 8, 0)
    ),
    (
        "C002",
        "Maghreb Logistics",
        "Standard",
        "Morocco",
        "Standard",
        True,
        datetime(2026, 1, 1, 8, 0)
    ),
    (
        "C003",
        "EuroFast",
        "Express",
        "France",
        "Premium",
        True,
        datetime(2026, 1, 1, 8, 0)
    ),
    (
        "C004",
        "NorthShip",
        "Standard",
        "Germany",
        "Standard",
        True,
        datetime(2026, 1, 1, 8, 0)
    ),
    (
        "C005",
        "RapidCargo",
        "Economy",
        "Spain",
        "Economy",
        True,
        datetime(2026, 1, 1, 8, 0)
    )
]

# ============================================================
# 2. SCHÉMA EXPLICITE
# ============================================================

carriers_schema = StructType([
    StructField("carrier_id", StringType(), False),
    StructField("carrier_name", StringType(), False),
    StructField("carrier_type", StringType(), False),
    StructField("country", StringType(), False),
    StructField("service_level", StringType(), False),
    StructField("active", BooleanType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 3. CRÉATION DU DATAFRAME
# ============================================================

carriers_df = spark.createDataFrame(
    carriers,
    schema=carriers_schema
)

# ============================================================
# 4. VÉRIFICATIONS
# ============================================================

print(carriers_df.columns)

carriers_df.printSchema()

display(carriers_df)

storage_account = "stsupplychain2026xx"

carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/carriers/"
)

(
    carriers_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(carriers_path)
)

print("Fichier carriers corrigé et réécrit dans la Landing Zone.")

display(
    dbutils.fs.ls(carriers_path)
)

In [0]:
carriers_df.printSchema()

In [0]:
print("Nombre de carriers :", carriers_df.count())
display(
    carriers_df.select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "service_level"
    )
)

In [0]:
from datetime import datetime

# ============================================================
# 1. Données source simulées : carriers
# ============================================================

carriers = [
    {
        "carrier_id": "C001",
        "carrier_name": "Atlas Express",
        "carrier_type": "Express",
        "country": "Morocco",
        "service_level": "Premium",
        "active": True,
        "updated_at": datetime(2026, 1, 1, 8, 0)
    },
    {
        "carrier_id": "C002",
        "carrier_name": "Maghreb Logistics",
        "carrier_type": "Standard",
        "country": "Morocco",
        "service_level": "Standard",
        "active": True,
        "updated_at": datetime(2026, 1, 1, 8, 0)
    },
    {
        "carrier_id": "C003",
        "carrier_name": "EuroFast",
        "carrier_type": "Express",
        "country": "France",
        "service_level": "Premium",
        "active": True,
        "updated_at": datetime(2026, 1, 1, 8, 0)
    },
    {
        "carrier_id": "C004",
        "carrier_name": "NorthShip",
        "carrier_type": "Standard",
        "country": "Germany",
        "service_level": "Standard",
        "active": True,
        "updated_at": datetime(2026, 1, 1, 8, 0)
    },
    {
        "carrier_id": "C005",
        "carrier_name": "RapidCargo",
        "carrier_type": "Economy",
        "country": "Spain",
        "service_level": "Economy",
        "active": True,
        "updated_at": datetime(2026, 1, 1, 8, 0)
    }
]

# ============================================================
# 2. Conversion en Spark DataFrame
# ============================================================

carriers_df = spark.createDataFrame(carriers)

# ============================================================
# 3. Vérifications
# ============================================================

display(carriers_df)

print("Nombre de carriers :", carriers_df.count())

carriers_df.printSchema()

# ============================================================
# 4. Chemin Landing Zone
# ============================================================

storage_account = "stsupplychain2026xx"

landing_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/carriers/"
)

# ============================================================
# 5. Écriture du fichier source CSV
# ============================================================

(
    carriers_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(landing_path)
)

# ============================================================
# 6. Vérification des fichiers créés
# ============================================================

files = dbutils.fs.ls(landing_path)

display(files)

In [0]:
import random
from datetime import datetime, timedelta
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(42)

storage_account = "stsupplychain2026xx"

orders_landing_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/orders/batch_001/"
)

# ============================================================
# 1. PARAMÈTRES DE GÉNÉRATION
# ============================================================

NUMBER_OF_ORDERS = 100

base_datetime = datetime(2026, 1, 1, 8, 0, 0)

cities = [
    ("Rabat", "Morocco"),
    ("Casablanca", "Morocco"),
    ("Marrakech", "Morocco"),
    ("Tangier", "Morocco"),
    ("Paris", "France"),
    ("Madrid", "Spain"),
    ("Berlin", "Germany")
]

order_statuses = [
    "CONFIRMED",
    "PROCESSING"
]

# ============================================================
# 2. GÉNÉRATION DES ORDERS
# ============================================================

orders = []

for i in range(1, NUMBER_OF_ORDERS + 1):

    order_id = f"O{i:04d}"

    customer_id = f"CUST{random.randint(1, 50):04d}"

    order_time = base_datetime + timedelta(
        minutes=random.randint(0, 3 * 24 * 60)
    )

    destination_city, destination_country = random.choice(cities)

    # IMPORTANT :
    # DecimalType Spark attend un decimal.Decimal Python
    total_amount = Decimal(
        str(round(random.uniform(100, 3000), 2))
    )

    currency = "MAD"

    order_status = random.choice(order_statuses)

    updated_at = order_time

    orders.append(
        (
            order_id,
            customer_id,
            order_time,
            destination_city,
            destination_country,
            total_amount,
            currency,
            order_status,
            updated_at
        )
    )

# ============================================================
# 3. SCHÉMA EXPLICITE
# ============================================================

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_time", TimestampType(), False),
    StructField("destination_city", StringType(), False),
    StructField("destination_country", StringType(), False),
    StructField("total_amount", DecimalType(10, 2), False),
    StructField("currency", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 4. CRÉATION DU DATAFRAME SPARK
# ============================================================

orders_df = spark.createDataFrame(
    orders,
    schema=orders_schema
)

# ============================================================
# 5. VÉRIFICATIONS
# ============================================================

print("Nombre de commandes :", orders_df.count())

orders_df.printSchema()

display(
    orders_df.orderBy("order_id")
)

# Vérification de l'unicité de order_id
duplicate_order_ids = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "Nombre de order_id dupliqués :",
    duplicate_order_ids
)

# ============================================================
# 6. ÉCRITURE DANS LA LANDING ZONE
# ============================================================

(
    orders_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(orders_landing_path)
)

# ============================================================
# 7. VÉRIFICATION DES FICHIERS ÉCRITS
# ============================================================

display(
    dbutils.fs.ls(orders_landing_path)
)

In [0]:
import random
from datetime import timedelta
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(43)

storage_account = "stsupplychain2026xx"

shipments_landing_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/shipments/batch_001/"
)

# ============================================================
# 1. RÉCUPÉRATION DES ORDERS DU BATCH 001
# ============================================================

orders_for_shipments = (
    orders_df
    .select(
        "order_id",
        "order_time",
        "destination_city"
    )
    .orderBy("order_id")
    .collect()
)

# ============================================================
# 2. PARAMÈTRES MÉTIER
# ============================================================

carrier_ids = [
    "C001",
    "C002",
    "C003",
    "C004",
    "C005"
]

origin_cities = [
    "Casablanca",
    "Rabat"
]

# ============================================================
# 3. GÉNÉRATION DES SHIPMENTS
# ============================================================

shipments = []

for index, order in enumerate(
    orders_for_shipments,
    start=1
):

    shipment_id = f"S{index:04d}"

    order_id = order["order_id"]
    order_time = order["order_time"]

    carrier_id = random.choice(carrier_ids)

    # L'expédition a lieu entre 1h et 24h après la commande
    shipment_time = (
        order_time
        + timedelta(
            hours=random.randint(1, 24)
        )
    )

    # Livraison prévue 1 à 5 jours après expédition
    expected_delivery_time = (
        shipment_time
        + timedelta(
            days=random.randint(1, 5)
        )
    )

    # Pas encore livré au moment du Batch 001
    actual_delivery_time = None

    origin_city = random.choice(origin_cities)

    destination_city = order["destination_city"]

    distance_km = Decimal(
        str(
            round(
                random.uniform(50, 2500),
                2
            )
        )
    )

    shipment_status = "CREATED"

    updated_at = shipment_time

    shipments.append(
        (
            shipment_id,
            order_id,
            carrier_id,
            shipment_time,
            expected_delivery_time,
            actual_delivery_time,
            origin_city,
            destination_city,
            distance_km,
            shipment_status,
            updated_at
        )
    )

# ============================================================
# 4. SCHÉMA EXPLICITE
# ============================================================

shipments_schema = StructType([
    StructField(
        "shipment_id",
        StringType(),
        False
    ),
    StructField(
        "order_id",
        StringType(),
        False
    ),
    StructField(
        "carrier_id",
        StringType(),
        False
    ),
    StructField(
        "shipment_time",
        TimestampType(),
        False
    ),
    StructField(
        "expected_delivery_time",
        TimestampType(),
        False
    ),
    StructField(
        "actual_delivery_time",
        TimestampType(),
        True
    ),
    StructField(
        "origin_city",
        StringType(),
        False
    ),
    StructField(
        "destination_city",
        StringType(),
        False
    ),
    StructField(
        "distance_km",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "shipment_status",
        StringType(),
        False
    ),
    StructField(
        "updated_at",
        TimestampType(),
        False
    )
])

# ============================================================
# 5. CRÉATION DU DATAFRAME
# ============================================================

shipments_df = spark.createDataFrame(
    shipments,
    schema=shipments_schema
)

# ============================================================
# 6. VÉRIFICATIONS
# ============================================================

print(
    "Nombre de shipments :",
    shipments_df.count()
)

shipments_df.printSchema()

display(
    shipments_df.orderBy("shipment_id")
)

# Vérification unicité shipment_id
duplicate_shipments = (
    shipments_df
    .groupBy("shipment_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "shipment_id dupliqués :",
    duplicate_shipments
)

# ============================================================
# 7. VÉRIFICATION DES FOREIGN KEYS
# ============================================================

invalid_order_references = (
    shipments_df.alias("s")
    .join(
        orders_df.alias("o"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

print(
    "Références order_id invalides :",
    invalid_order_references
)

invalid_carrier_references = (
    shipments_df.alias("s")
    .join(
        carriers_df.alias("c"),
        on="carrier_id",
        how="left_anti"
    )
    .count()
)

print(
    "Références carrier_id invalides :",
    invalid_carrier_references
)

# ============================================================
# 8. VÉRIFICATION TEMPORELLE
# ============================================================

invalid_shipment_times = (
    shipments_df.alias("s")
    .join(
        orders_df.select(
            "order_id",
            "order_time"
        ).alias("o"),
        on="order_id",
        how="inner"
    )
    .filter(
        "shipment_time <= order_time"
    )
    .count()
)

print(
    "Expéditions temporellement invalides :",
    invalid_shipment_times
)

# ============================================================
# 9. ÉCRITURE LANDING ZONE
# ============================================================

(
    shipments_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(shipments_landing_path)
)

# ============================================================
# 10. VÉRIFICATION DES FICHIERS
# ============================================================

display(
    dbutils.fs.ls(
        shipments_landing_path
    )
)

In [0]:
import random
from datetime import timedelta

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(44)

storage_account = "stsupplychain2026xx"

events_landing_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/delivery_events/batch_001/"
)

# ============================================================
# 1. RÉCUPÉRATION DES SHIPMENTS DU BATCH 001
# ============================================================

shipments_for_events = (
    shipments_df
    .select(
        "shipment_id",
        "shipment_time",
        "origin_city",
        "destination_city"
    )
    .orderBy("shipment_id")
    .collect()
)

# ============================================================
# 2. STATUTS DU LIFECYCLE
# ============================================================

event_sequence = [
    "CREATED",
    "PACKED",
    "SHIPPED",
    "HUB_SCAN",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

# ============================================================
# 3. GÉNÉRATION DES DELIVERY EVENTS
# ============================================================

delivery_events = []

event_counter = 1

for shipment in shipments_for_events:

    shipment_id = shipment["shipment_id"]
    shipment_time = shipment["shipment_time"]
    origin_city = shipment["origin_city"]
    destination_city = shipment["destination_city"]

    # --------------------------------------------------------
    # CREATED
    # --------------------------------------------------------

    created_time = shipment_time

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            created_time,
            "CREATED",
            origin_city,
            "Morocco",
            "logistics_app",
            created_time
        )
    )

    event_counter += 1

    # --------------------------------------------------------
    # PACKED
    # --------------------------------------------------------

    packed_time = (
        created_time
        + timedelta(
            hours=random.randint(1, 4)
        )
    )

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            packed_time,
            "PACKED",
            origin_city,
            "Morocco",
            "warehouse_system",
            packed_time
        )
    )

    event_counter += 1

    # --------------------------------------------------------
    # SHIPPED
    # --------------------------------------------------------

    shipped_time = (
        packed_time
        + timedelta(
            hours=random.randint(1, 5)
        )
    )

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            shipped_time,
            "SHIPPED",
            origin_city,
            "Morocco",
            "carrier_system",
            shipped_time
        )
    )

    event_counter += 1

    # --------------------------------------------------------
    # HUB_SCAN
    # --------------------------------------------------------

    hub_scan_time = (
        shipped_time
        + timedelta(
            hours=random.randint(4, 24)
        )
    )

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            hub_scan_time,
            "HUB_SCAN",
            "Transit Hub",
            "Morocco",
            "carrier_system",
            hub_scan_time
        )
    )

    event_counter += 1

    # --------------------------------------------------------
    # OUT_FOR_DELIVERY
    # --------------------------------------------------------

    out_for_delivery_time = (
        hub_scan_time
        + timedelta(
            hours=random.randint(4, 24)
        )
    )

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            out_for_delivery_time,
            "OUT_FOR_DELIVERY",
            destination_city,
            "Morocco",
            "carrier_system",
            out_for_delivery_time
        )
    )

    event_counter += 1

    # --------------------------------------------------------
    # DELIVERED
    # --------------------------------------------------------

    delivered_time = (
        out_for_delivery_time
        + timedelta(
            hours=random.randint(1, 8)
        )
    )

    delivery_events.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            delivered_time,
            "DELIVERED",
            destination_city,
            "Morocco",
            "carrier_system",
            delivered_time
        )
    )

    event_counter += 1

# ============================================================
# 4. SCHÉMA EXPLICITE
# ============================================================

delivery_events_schema = StructType([
    StructField(
        "event_id",
        StringType(),
        False
    ),
    StructField(
        "shipment_id",
        StringType(),
        False
    ),
    StructField(
        "event_time",
        TimestampType(),
        False
    ),
    StructField(
        "event_status",
        StringType(),
        False
    ),
    StructField(
        "location_city",
        StringType(),
        True
    ),
    StructField(
        "location_country",
        StringType(),
        True
    ),
    StructField(
        "source_system",
        StringType(),
        False
    ),
    StructField(
        "updated_at",
        TimestampType(),
        False
    )
])

# ============================================================
# 5. CRÉATION DU DATAFRAME
# ============================================================

delivery_events_df = spark.createDataFrame(
    delivery_events,
    schema=delivery_events_schema
)

# ============================================================
# 6. VÉRIFICATIONS
# ============================================================

print(
    "Nombre de delivery events :",
    delivery_events_df.count()
)

delivery_events_df.printSchema()

display(
    delivery_events_df.orderBy(
        "shipment_id",
        "event_time"
    )
)

# ============================================================
# 7. VÉRIFICATION UNICITÉ EVENT_ID
# ============================================================

duplicate_event_ids = (
    delivery_events_df
    .groupBy("event_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "event_id dupliqués :",
    duplicate_event_ids
)

# ============================================================
# 8. VÉRIFICATION DES SHIPMENT_ID
# ============================================================

invalid_shipment_references = (
    delivery_events_df
    .join(
        shipments_df.select("shipment_id"),
        on="shipment_id",
        how="left_anti"
    )
    .count()
)

print(
    "Références shipment_id invalides :",
    invalid_shipment_references
)

# ============================================================
# 9. VÉRIFICATION DU NOMBRE D'ÉVÉNEMENTS PAR SHIPMENT
# ============================================================

events_per_shipment = (
    delivery_events_df
    .groupBy("shipment_id")
    .count()
)

display(
    events_per_shipment.orderBy("shipment_id")
)

invalid_event_counts = (
    events_per_shipment
    .filter("count != 6")
    .count()
)

print(
    "Shipments avec un nombre d'événements incorrect :",
    invalid_event_counts
)

# ============================================================
# 10. ÉCRITURE DANS LA LANDING ZONE
# ============================================================

(
    delivery_events_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(events_landing_path)
)

# ============================================================
# 11. VÉRIFICATION DES FICHIERS ÉCRITS
# ============================================================

display(
    dbutils.fs.ls(
        events_landing_path
    )
)

In [0]:
import random
from datetime import datetime, timedelta
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(45)

storage_account = "stsupplychain2026xx"

orders_batch_002_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/orders/batch_002/"
)

# ============================================================
# 1. PARAMÈTRES DU BATCH 002
# ============================================================

NUMBER_OF_NEW_ORDERS = 30

# Batch 001 contenait O0001 -> O0100
START_ORDER_ID = 101

base_datetime_batch_002 = datetime(
    2026,
    1,
    5,
    8,
    0,
    0
)

cities = [
    ("Rabat", "Morocco"),
    ("Casablanca", "Morocco"),
    ("Marrakech", "Morocco"),
    ("Tangier", "Morocco"),
    ("Paris", "France"),
    ("Madrid", "Spain"),
    ("Berlin", "Germany")
]

order_statuses = [
    "CONFIRMED",
    "PROCESSING"
]

# ============================================================
# 2. GÉNÉRATION DU BATCH 002
# ============================================================

orders_batch_002 = []

for i in range(
    START_ORDER_ID,
    START_ORDER_ID + NUMBER_OF_NEW_ORDERS
):

    order_id = f"O{i:04d}"

    customer_id = (
        f"CUST{random.randint(1, 50):04d}"
    )

    order_time = (
        base_datetime_batch_002
        + timedelta(
            minutes=random.randint(
                0,
                2 * 24 * 60
            )
        )
    )

    destination_city, destination_country = (
        random.choice(cities)
    )

    total_amount = Decimal(
        str(
            round(
                random.uniform(
                    100,
                    3000
                ),
                2
            )
        )
    )

    currency = "MAD"

    order_status = random.choice(
        order_statuses
    )

    updated_at = order_time

    orders_batch_002.append(
        (
            order_id,
            customer_id,
            order_time,
            destination_city,
            destination_country,
            total_amount,
            currency,
            order_status,
            updated_at
        )
    )

# ============================================================
# 3. SCHÉMA
# ============================================================

orders_schema = StructType([
    StructField(
        "order_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "order_time",
        TimestampType(),
        False
    ),
    StructField(
        "destination_city",
        StringType(),
        False
    ),
    StructField(
        "destination_country",
        StringType(),
        False
    ),
    StructField(
        "total_amount",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "order_status",
        StringType(),
        False
    ),
    StructField(
        "updated_at",
        TimestampType(),
        False
    )
])

# ============================================================
# 4. DATAFRAME
# ============================================================

orders_batch_002_df = (
    spark.createDataFrame(
        orders_batch_002,
        schema=orders_schema
    )
)

# ============================================================
# 5. VÉRIFICATIONS
# ============================================================

print(
    "Nombre de nouvelles commandes :",
    orders_batch_002_df.count()
)

print(
    "Premier order_id :",
    orders_batch_002_df
    .orderBy("order_id")
    .first()["order_id"]
)

print(
    "Dernier order_id :",
    orders_batch_002_df
    .orderBy(
        "order_id",
        ascending=False
    )
    .first()["order_id"]
)

display(
    orders_batch_002_df
    .orderBy("order_id")
)

# ============================================================
# 6. ÉCRITURE DANS LA LANDING ZONE
# ============================================================

(
    orders_batch_002_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option(
        "header",
        "true"
    )
    .csv(
        orders_batch_002_path
    )
)

# ============================================================
# 7. VÉRIFICATION DES FICHIERS
# ============================================================

display(
    dbutils.fs.ls(
        orders_batch_002_path
    )
)

print(
    "Batch 002 orders généré avec succès."
)

In [0]:
import random
from datetime import timedelta
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(46)

storage_account = "stsupplychain2026xx"

shipments_batch_002_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/shipments/batch_002/"
)

# ============================================================
# 1. RÉCUPÉRER LES ORDERS DU BATCH 002
# ============================================================

orders_batch_002_for_shipments = (
    orders_batch_002_df
    .select(
        "order_id",
        "order_time",
        "destination_city"
    )
    .orderBy("order_id")
    .collect()
)

# ============================================================
# 2. PARAMÈTRES
# ============================================================

carrier_ids = [
    "C001",
    "C002",
    "C003",
    "C004",
    "C005"
]

origin_cities = [
    "Casablanca",
    "Rabat"
]

# ============================================================
# 3. GÉNÉRATION DES SHIPMENTS BATCH 002
# ============================================================

shipments_batch_002 = []

for index, order in enumerate(
    orders_batch_002_for_shipments,
    start=101
):

    shipment_id = f"S{index:04d}"

    order_id = order["order_id"]
    order_time = order["order_time"]

    carrier_id = random.choice(carrier_ids)

    shipment_time = (
        order_time
        + timedelta(
            hours=random.randint(1, 24)
        )
    )

    expected_delivery_time = (
        shipment_time
        + timedelta(
            days=random.randint(1, 5)
        )
    )

    actual_delivery_time = None

    origin_city = random.choice(origin_cities)

    destination_city = order["destination_city"]

    distance_km = Decimal(
        str(
            round(
                random.uniform(50, 2500),
                2
            )
        )
    )

    shipment_status = "CREATED"

    updated_at = shipment_time

    shipments_batch_002.append(
        (
            shipment_id,
            order_id,
            carrier_id,
            shipment_time,
            expected_delivery_time,
            actual_delivery_time,
            origin_city,
            destination_city,
            distance_km,
            shipment_status,
            updated_at
        )
    )

# ============================================================
# 4. SCHÉMA
# ============================================================

shipments_schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("carrier_id", StringType(), False),
    StructField("shipment_time", TimestampType(), False),
    StructField("expected_delivery_time", TimestampType(), False),
    StructField("actual_delivery_time", TimestampType(), True),
    StructField("origin_city", StringType(), False),
    StructField("destination_city", StringType(), False),
    StructField("distance_km", DecimalType(10, 2), False),
    StructField("shipment_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 5. DATAFRAME
# ============================================================

shipments_batch_002_df = spark.createDataFrame(
    shipments_batch_002,
    schema=shipments_schema
)

# ============================================================
# 6. VÉRIFICATIONS
# ============================================================

print(
    "Nombre de nouveaux shipments :",
    shipments_batch_002_df.count()
)

display(
    shipments_batch_002_df
    .orderBy("shipment_id")
)

# Vérifier que chaque order_id existe bien dans batch_002
invalid_orders = (
    shipments_batch_002_df
    .join(
        orders_batch_002_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

print(
    "Références order_id invalides :",
    invalid_orders
)

# ============================================================
# 7. ÉCRITURE LANDING
# ============================================================

(
    shipments_batch_002_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(shipments_batch_002_path)
)

display(
    dbutils.fs.ls(
        shipments_batch_002_path
    )
)

In [0]:
import random
from datetime import timedelta

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

# ============================================================
# CONFIGURATION
# ============================================================

random.seed(47)

storage_account = "stsupplychain2026xx"

events_batch_002_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/delivery_events/batch_002/"
)

# ============================================================
# 1. RÉCUPÉRER LES SHIPMENTS BATCH 002
# ============================================================

shipments_batch_002_for_events = (
    shipments_batch_002_df
    .select(
        "shipment_id",
        "shipment_time",
        "origin_city",
        "destination_city"
    )
    .orderBy("shipment_id")
    .collect()
)

# ============================================================
# 2. GÉNÉRATION DES EVENTS
# ============================================================

delivery_events_batch_002 = []

event_counter = 601

for shipment in shipments_batch_002_for_events:

    shipment_id = shipment["shipment_id"]
    shipment_time = shipment["shipment_time"]
    origin_city = shipment["origin_city"]
    destination_city = shipment["destination_city"]

    created_time = shipment_time

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            created_time,
            "CREATED",
            origin_city,
            "Morocco",
            "logistics_app",
            created_time
        )
    )
    event_counter += 1

    packed_time = (
        created_time
        + timedelta(
            hours=random.randint(1, 4)
        )
    )

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            packed_time,
            "PACKED",
            origin_city,
            "Morocco",
            "warehouse_system",
            packed_time
        )
    )
    event_counter += 1

    shipped_time = (
        packed_time
        + timedelta(
            hours=random.randint(1, 5)
        )
    )

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            shipped_time,
            "SHIPPED",
            origin_city,
            "Morocco",
            "carrier_system",
            shipped_time
        )
    )
    event_counter += 1

    hub_scan_time = (
        shipped_time
        + timedelta(
            hours=random.randint(4, 24)
        )
    )

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            hub_scan_time,
            "HUB_SCAN",
            "Transit Hub",
            "Morocco",
            "carrier_system",
            hub_scan_time
        )
    )
    event_counter += 1

    out_for_delivery_time = (
        hub_scan_time
        + timedelta(
            hours=random.randint(4, 24)
        )
    )

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            out_for_delivery_time,
            "OUT_FOR_DELIVERY",
            destination_city,
            "Morocco",
            "carrier_system",
            out_for_delivery_time
        )
    )
    event_counter += 1

    delivered_time = (
        out_for_delivery_time
        + timedelta(
            hours=random.randint(1, 8)
        )
    )

    delivery_events_batch_002.append(
        (
            f"E{event_counter:06d}",
            shipment_id,
            delivered_time,
            "DELIVERED",
            destination_city,
            "Morocco",
            "carrier_system",
            delivered_time
        )
    )
    event_counter += 1

# ============================================================
# 3. SCHÉMA
# ============================================================

delivery_events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("shipment_id", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("event_status", StringType(), False),
    StructField("location_city", StringType(), True),
    StructField("location_country", StringType(), True),
    StructField("source_system", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 4. DATAFRAME
# ============================================================

delivery_events_batch_002_df = (
    spark.createDataFrame(
        delivery_events_batch_002,
        schema=delivery_events_schema
    )
)

# ============================================================
# 5. VÉRIFICATIONS
# ============================================================

print(
    "Nombre de nouveaux events :",
    delivery_events_batch_002_df.count()
)

invalid_shipments = (
    delivery_events_batch_002_df
    .join(
        shipments_batch_002_df.select("shipment_id"),
        on="shipment_id",
        how="left_anti"
    )
    .count()
)

print(
    "Références shipment_id invalides :",
    invalid_shipments
)

# ============================================================
# 6. ÉCRITURE LANDING
# ============================================================

(
    delivery_events_batch_002_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(events_batch_002_path)
)

display(
    dbutils.fs.ls(
        events_batch_002_path
    )
)

In [0]:
from datetime import datetime
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

orders_batch_003_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/orders/batch_003/"
)

# ============================================================
# 1. DONNÉES DU BATCH 003
# ============================================================

orders_batch_003 = [

    # --------------------------------------------------------
    # CAS 1 : nouvelle commande parfaitement valide
    # --------------------------------------------------------
    (
        "O0131",
        "CUST0010",
        datetime(2026, 1, 10, 8, 0),
        "Rabat",
        "Morocco",
        Decimal("1250.00"),
        "MAD",
        "CONFIRMED",
        datetime(2026, 1, 10, 8, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : montant invalide
    # --------------------------------------------------------
    (
        "O0132",
        "CUST0011",
        datetime(2026, 1, 10, 9, 0),
        "Casablanca",
        "Morocco",
        Decimal("-500.00"),
        "MAD",
        "CONFIRMED",
        datetime(2026, 1, 10, 9, 0)
    ),

    # --------------------------------------------------------
    # CAS 3 : statut métier invalide
    # --------------------------------------------------------
    (
        "O0133",
        "CUST0012",
        datetime(2026, 1, 10, 10, 0),
        "Marrakech",
        "Morocco",
        Decimal("850.00"),
        "MAD",
        "UNKNOWN_STATUS",
        datetime(2026, 1, 10, 10, 0)
    ),

    # --------------------------------------------------------
    # CAS 4 : devise invalide pour notre règle métier
    # --------------------------------------------------------
    (
        "O0134",
        "CUST0013",
        datetime(2026, 1, 10, 11, 0),
        "Tangier",
        "Morocco",
        Decimal("950.00"),
        "EUR",
        "PROCESSING",
        datetime(2026, 1, 10, 11, 0)
    ),

    # --------------------------------------------------------
    # CAS 5 : correction d'une commande existante
    #
    # O0050 existe déjà dans un ancien batch.
    # updated_at est plus récent : cette version devra gagner
    # lors de la déduplication Silver.
    # --------------------------------------------------------
    (
        "O0050",
        "CUST0025",
        datetime(2026, 1, 2, 12, 0),
        "Rabat",
        "Morocco",
        Decimal("2200.00"),
        "MAD",
        "PROCESSING",
        datetime(2026, 1, 10, 12, 0)
    )
]

# ============================================================
# 2. SCHÉMA
# ============================================================

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_time", TimestampType(), False),
    StructField("destination_city", StringType(), False),
    StructField("destination_country", StringType(), False),
    StructField("total_amount", DecimalType(10, 2), False),
    StructField("currency", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 3. DATAFRAME
# ============================================================

orders_batch_003_df = spark.createDataFrame(
    orders_batch_003,
    schema=orders_schema
)

# ============================================================
# 4. VÉRIFICATION
# ============================================================

print(
    "Nombre de lignes Batch 003 :",
    orders_batch_003_df.count()
)

display(
    orders_batch_003_df.orderBy("order_id")
)

# ============================================================
# 5. ÉCRITURE DANS LA LANDING ZONE
# ============================================================

(
    orders_batch_003_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(orders_batch_003_path)
)

print("Batch 003 orders créé.")

display(
    dbutils.fs.ls(orders_batch_003_path)
)

In [0]:
from datetime import datetime
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

shipments_batch_003_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/shipments/batch_003/"
)

# ============================================================
# 1. BATCH 003 SHIPMENTS
# ============================================================

shipments_batch_003 = [

    # --------------------------------------------------------
    # CAS 1 : shipment valide
    # --------------------------------------------------------
    (
        "S0131",
        "O0131",
        "C001",
        datetime(2026, 1, 10, 10, 0),
        datetime(2026, 1, 12, 10, 0),
        None,
        "Rabat",
        "Rabat",
        Decimal("25.00"),
        "CREATED",
        datetime(2026, 1, 10, 10, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : référence vers un order invalide
    # --------------------------------------------------------
    (
        "S0132",
        "O0132",
        "C002",
        datetime(2026, 1, 10, 11, 0),
        datetime(2026, 1, 13, 11, 0),
        None,
        "Casablanca",
        "Casablanca",
        Decimal("30.00"),
        "CREATED",
        datetime(2026, 1, 10, 11, 0)
    ),

    # --------------------------------------------------------
    # CAS 3 : carrier inexistant
    # --------------------------------------------------------
    (
        "S0133",
        "O0130",
        "C999",
        datetime(2026, 1, 10, 12, 0),
        datetime(2026, 1, 14, 12, 0),
        None,
        "Casablanca",
        "Rabat",
        Decimal("90.00"),
        "CREATED",
        datetime(2026, 1, 10, 12, 0)
    ),

    # --------------------------------------------------------
    # CAS 4 : incohérence temporelle volontaire
    # --------------------------------------------------------
    (
        "S0134",
        "O0129",
        "C003",
        datetime(2026, 1, 10, 13, 0),
        datetime(2026, 1, 9, 13, 0),
        None,
        "Rabat",
        "Casablanca",
        Decimal("95.00"),
        "CREATED",
        datetime(2026, 1, 10, 13, 0)
    ),

    # --------------------------------------------------------
    # CAS 5 : nouvelle version de S0050
    #
    # IMPORTANT :
    # on conserve le shipment_time historique original
    # pour ne pas casser les anciens événements.
    # --------------------------------------------------------
    (
        "S0050",
        "O0050",
        "C001",
        datetime(2026, 1, 2, 3, 22),   # valeur corrigée
        datetime(2026, 1, 6, 14, 0),
        None,
        "Casablanca",
        "Rabat",
        Decimal("110.00"),
        "SHIPPED",
        datetime(2026, 1, 10, 14, 0)
    )
]

# ============================================================
# 2. SCHÉMA
# ============================================================

shipments_schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("carrier_id", StringType(), False),
    StructField("shipment_time", TimestampType(), False),
    StructField("expected_delivery_time", TimestampType(), False),
    StructField("actual_delivery_time", TimestampType(), True),
    StructField("origin_city", StringType(), False),
    StructField("destination_city", StringType(), False),
    StructField("distance_km", DecimalType(10, 2), False),
    StructField("shipment_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 3. DATAFRAME
# ============================================================

shipments_batch_003_df = spark.createDataFrame(
    shipments_batch_003,
    schema=shipments_schema
)

# ============================================================
# 4. VÉRIFICATION
# ============================================================

print(
    "Nombre de lignes Batch 003 :",
    shipments_batch_003_df.count()
)

display(
    shipments_batch_003_df
    .orderBy("shipment_id")
)

# ============================================================
# 5. ÉCRITURE LANDING
# ============================================================

(
    shipments_batch_003_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(shipments_batch_003_path)
)

print("Batch 003 shipments corrigé et réécrit.")

display(
    dbutils.fs.ls(
        shipments_batch_003_path
    )
)

In [0]:
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

# ============================================================
# CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

events_batch_003_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/delivery_events/batch_003/"
)

# ============================================================
# 1. BATCH 003 EVENTS
# ============================================================

delivery_events_batch_003 = [

    # --------------------------------------------------------
    # CAS 1 : event valide pour S0131
    # --------------------------------------------------------
    (
        "E000781",
        "S0131",
        datetime(2026, 1, 10, 10, 0),
        "CREATED",
        "Rabat",
        "Morocco",
        "logistics_app",
        datetime(2026, 1, 10, 10, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : shipment inexistant
    # --------------------------------------------------------
    (
        "E000782",
        "S9999",
        datetime(2026, 1, 10, 12, 0),
        "CREATED",
        "Rabat",
        "Morocco",
        "logistics_app",
        datetime(2026, 1, 10, 12, 0)
    ),

    # --------------------------------------------------------
    # CAS 3 : statut invalide
    # --------------------------------------------------------
    (
        "E000783",
        "S0131",
        datetime(2026, 1, 10, 13, 0),
        "UNKNOWN_EVENT",
        "Rabat",
        "Morocco",
        "carrier_system",
        datetime(2026, 1, 10, 13, 0)
    ),

    # --------------------------------------------------------
    # CAS 4 : correction d'un event existant
    #
    # On réutilise un event_id déjà présent.
    # updated_at est plus récent.
    # --------------------------------------------------------
    (
        "E000050",
        "S0009",
        datetime(2026, 1, 4, 10, 43),  # event_time historique cohérent
        "PACKED",
        "Casablanca",
        "Morocco",
        "warehouse_system",
        datetime(2026, 1, 10, 15, 0)   # correction reçue plus tard
    ),

    # --------------------------------------------------------
    # CAS 5 : LATE-ARRIVING EVENT
    #
    # Cet événement PACKED a eu lieu à 14:00.
    # Mais il arrive dans un batch ultérieur,
    # donc après d'autres événements déjà ingérés.
    #
    # Il doit rester VALIDE car le lifecycle repose
    # sur event_time, pas sur ingestion_timestamp.
    # --------------------------------------------------------
    (
        "E000784",
        "S0131",
        datetime(2026, 1, 10, 14, 0),
        "PACKED",
        "Rabat",
        "Morocco",
        "warehouse_system",
        datetime(2026, 1, 10, 18, 0)
    ),

    # --------------------------------------------------------
    # CAS 6 : SHIPPED pour S0131
    #
    # event_time plus tard que PACKED,
    # mais on imagine qu'il a été reçu plus tôt.
    # --------------------------------------------------------
    (
        "E000785",
        "S0131",
        datetime(2026, 1, 10, 16, 0),
        "SHIPPED",
        "Rabat",
        "Morocco",
        "carrier_system",
        datetime(2026, 1, 10, 16, 30)
    )
]

# ============================================================
# 2. SCHÉMA
# ============================================================

delivery_events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("shipment_id", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("event_status", StringType(), False),
    StructField("location_city", StringType(), True),
    StructField("location_country", StringType(), True),
    StructField("source_system", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

# ============================================================
# 3. DATAFRAME
# ============================================================

delivery_events_batch_003_df = spark.createDataFrame(
    delivery_events_batch_003,
    schema=delivery_events_schema
)

# ============================================================
# 4. VÉRIFICATION
# ============================================================

print(
    "Nombre de lignes Batch 003 :",
    delivery_events_batch_003_df.count()
)

display(
    delivery_events_batch_003_df
    .orderBy("event_id")
)

# ============================================================
# 5. ÉCRITURE LANDING
# ============================================================

(
    delivery_events_batch_003_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(events_batch_003_path)
)

print("Batch 003 delivery_events créé.")

display(
    dbutils.fs.ls(
        events_batch_003_path
    )
)